<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####Multilevel Aggregates
Multilevel aggregates allows summarizing data at several hierarchical levels.\
We have three variations of multilevel aggregates in Spark.
1. Rollup
2. Cube
3. Grouping Sets



####Requirement - Analysis Data Set

Prepare club bookings dataset for analysis
```
+----------+------------+---------------+-------------------+--------------+
|booking_id| member_name|  facility_name|         start_time|booking_amount|
+----------+------------+---------------+-------------------+--------------+
```

In [0]:
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

club_bookings_df = (
    bookings_df.join(facilities_df, "facility_id")
            .join(members_df, "member_id", "left")
            .selectExpr("member_id", "booking_id",
                        "case when member_id==0 then 'Guest Member' else concat_ws(' ', first_name, last_name) end as member_name",
                        "facility_name","start_time",
                        "case when member_id == 0 then slots * guest_cost else slots * member_cost end as booking_amount")
)

club_bookings_df.display()

member_id,booking_id,member_name,facility_name,start_time,booking_amount
1,0,Darren Smith,Table Tennis,2022-07-03T11:00:00.000Z,0.0
1,1,Darren Smith,Massage Room 1,2022-07-03T08:00:00.000Z,70.0
0,2,Guest Member,Squash Court,2022-07-03T18:00:00.000Z,35.0
1,3,Darren Smith,Snooker Table,2022-07-03T19:00:00.000Z,0.0
1,4,Darren Smith,Pool Table,2022-07-03T10:00:00.000Z,0.0
1,5,Darren Smith,Pool Table,2022-07-03T15:00:00.000Z,0.0
2,6,Tracy Smith,Tennis Court 1,2022-07-04T09:00:00.000Z,15.0
2,7,Tracy Smith,Tennis Court 1,2022-07-04T15:00:00.000Z,15.0
3,8,Tim Rownam,Massage Room 1,2022-07-04T13:30:00.000Z,70.0
0,9,Guest Member,Massage Room 1,2022-07-04T15:00:00.000Z,160.0


Q1. Prepare a monthly revenue report for year 2022.

Also roll up the total for the month column.
```
+----+--------+
|mnth| revenue|
+----+--------+
|   7| 23202.5|
|   8| 46066.5|
|   9| 63315.5|
|    |132584.5|
+----+--------+
```

In [0]:
from pyspark.sql.functions import month, sum, col

result_df = (
    club_bookings_df.where("year(start_time) == 2022")
        .withColumn("mnth", month("start_time"))
        .rollup("mnth")
        .agg(sum("booking_amount").alias("revenue"))
        .orderBy(col("mnth").asc_nulls_last())
)

result_df.display()

mnth,revenue
7,23202.5
8,46066.5
9,63315.5
null,132584.5


Q2. Prepare a revenue report by revenue_from (Guest/Member) and facility_name for year 2022.

Also roll up the total for each group.
```
+------------+---------------+-------+
|revenue_from|  facility_name|revenue|
+------------+---------------+-------+
|       Guest|Badminton Court| 1906.5|
|       Guest| Massage Room 1|41600.0|
|       Guest| ..............|.......|
|       Guest|     ..........|  .....|
|       Guest|           NULL|89096.5|
|      Member|Badminton Court|    0.0|
|      Member| Massage Room 1|30940.0|
|      Member| ..............| ......|
|      Member| ..............| ......|
|      Member|           NULL|43488.0|
+------------+---------------+-------+
```

In [0]:
from pyspark.sql.functions import sum, col, expr

result_df = (
    club_bookings_df.where("year(start_time) == 2022")
        .withColumn("revenue_from", expr("case when member_id==0 then 'Guest' else 'Member' end"))
        .rollup("revenue_from", "facility_name")
        .agg(sum("booking_amount").alias("revenue"))
        .orderBy(col("revenue_from").asc_nulls_last(),
                 col("facility_name").asc_nulls_last())
)

result_df.display()

revenue_from,facility_name,revenue
Guest,Badminton Court,1906.5
Guest,Massage Room 1,41600.0
Guest,Massage Room 2,13920.0
Guest,Pool Table,270.0
Guest,Snooker Table,240.0
Guest,Squash Court,12005.0
Guest,Table Tennis,180.0
Guest,Tennis Court 1,9075.0
Guest,Tennis Court 2,9900.0
Guest,null,89096.5


Q3. Prepare a revenue report by revenue_from(Guest/Member) and facility_name for year 2022.

Also compute totals for all 4 dimensions of revenue_from and facility_name.
* (revenue_from, facility_name) : rollup
* (revenue_from, ) : rollup
* (facility_name, ) : not available in roolup
* ( , ) : grand total in rollup

In [0]:
from pyspark.sql.functions import sum, col, expr

result_df = (
    club_bookings_df.where("year(start_time) == 2022")
        .withColumn("revenue_from", expr("case when member_id==0 then 'Guest' else 'Member' end"))
        .cube("revenue_from", "facility_name")
        .agg(sum("booking_amount").alias("revenue"))
        .orderBy(col("revenue_from").asc_nulls_last(),
                 col("facility_name").asc_nulls_last())
)

result_df.display()

revenue_from,facility_name,revenue
Guest,Badminton Court,1906.5
Guest,Massage Room 1,41600.0
Guest,Massage Room 2,13920.0
Guest,Pool Table,270.0
Guest,Snooker Table,240.0
Guest,Squash Court,12005.0
Guest,Table Tennis,180.0
Guest,Tennis Court 1,9075.0
Guest,Tennis Court 2,9900.0
Guest,null,89096.5


Q4: Prepare a revenue report similar to the following.
```
  revenue_from  | facility_name   | revenue
  ---------------------------------------
  Guest         | Badminton Court | 1906.5
  Guest         | Massage Room 1  | 41600
  Guest         | Massage Room 2  | 13920
  Guest         |                 | 57426.5
  Member        | Badminton Court | 0
  Member        | Massage Room 1  | 30940
  Member        | Massage Room 2  | 1890
  Member        |                 | 32830
```
Roll up the total for a the facility_name only.


In [0]:
from pyspark.sql.functions import sum, col, expr

result_df = (
    club_bookings_df.where("year(start_time) == 2022")
        .withColumn("revenue_from", expr("case when member_id==0 then 'Guest' else 'Member' end"))
        .groupingSets([("revenue_from","facility_name"), ("revenue_from", )], "revenue_from", "facility_name")
        .agg(sum("booking_amount").alias("revenue"))
        .orderBy(col("revenue_from").asc_nulls_last(),
                 col("facility_name").asc_nulls_last())
)

result_df.display()

revenue_from,facility_name,revenue
Guest,Badminton Court,1906.5
Guest,Massage Room 1,41600.0
Guest,Massage Room 2,13920.0
Guest,Pool Table,270.0
Guest,Snooker Table,240.0
Guest,Squash Court,12005.0
Guest,Table Tennis,180.0
Guest,Tennis Court 1,9075.0
Guest,Tennis Court 2,9900.0
Guest,null,89096.5


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>